<div style="direction: rtl; white-space: normal; line-height: 1;">
# Clean Quran Dataset

هدف این مرحله:

- خواندن دیتاست آیه‌محور
- نرمال‌سازی متن‌ها برای پردازش NLP
- حفظ متن اصلی قرآن و ترجمه‌ها
- ساخت نسخه آماده برای embedding
</div>

In [15]:
from pathlib import Path
import json
import re


# Project root
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


# Data paths
processed_dir = project_root / "data" / "processed"

input_file = processed_dir / "quran_dataset.json"
output_file = processed_dir / "quran_dataset_clean.json"


print("Input:", input_file)
print("Output:", output_file)

Input: /Users/macbookpro/Desktop/_PROGRAMING/QuranRAG/data/processed/quran_dataset.json
Output: /Users/macbookpro/Desktop/_PROGRAMING/QuranRAG/data/processed/quran_dataset_clean.json


<div style="direction: rtl; white-space: normal; line-height: 1;">
## Text cleaning functions

در این بخش توابع نرمال‌سازی متن را تعریف می‌کنیم.

توجه:
- متن اصلی قرآن تغییر نمی‌کند.
- این توابع فقط برای ساخت نسخه پردازشی استفاده می‌شوند.
</div>

In [17]:
# Text normalization functions

def normalize_arabic_text(text):
    """
    Normalize Arabic/Persian characters
    """
    text = text.replace("ي", "ی")
    text = text.replace("ى", "ی")
    text = text.replace("ك", "ک")

    return text


def clean_text(text):
    """
    General text cleaning
    """
    text = normalize_arabic_text(text)

    # Remove hidden characters
    text = re.sub(r"\s+", " ", text)

    # Remove extra spaces
    text = text.strip()

    return text


def remove_diacritics(text):
    """
    Remove Arabic diacritics for search only
    """
    arabic_diacritics = re.compile(
        r'[\u0617-\u061A\u064B-\u0652\u0670\u06D6-\u06ED]'
    )

    text = arabic_diacritics.sub('', text)

    # Normalize Arabic special letters for simple search
    text = text.replace("ٱ", "ا")

    return text


print("Cleaning functions are ready.")

Cleaning functions are ready.


<div style="direction: rtl; white-space: normal; line-height: 1;">
## Load dataset and create cleaned version

در این بخش:
- فایل JSON اصلی خوانده می‌شود.
- برای هر آیه فیلدهای clean ساخته می‌شوند.
- متن اصلی حفظ می‌شود.
</div>

In [18]:
# Load Quran dataset

with open(input_file, "r", encoding="utf-8") as f:
    quran_data = json.load(f)


print("Loaded records:", len(quran_data))
print(quran_data[0])

Loaded records: 6236
{'surah': 1, 'ayah': 1, 'arabic': 'بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ', 'fooladvand': 'به نام خداوند رحمتگر مهربان', 'ansarian': 'به نام خدا که رحمتش بی\u200cاندازه است و مهربانی\u200cاش همیشگی.'}


<div style="direction: rtl; white-space: normal; line-height: 1;">
## Create cleaned Quran records

در این بخش:
- اطلاعات اصلی حفظ می‌شود.
- فیلدهای جدید clean برای جستجو و embedding ساخته می‌شوند.
</div>

In [19]:
# Create cleaned dataset

clean_records = []

for item in quran_data:
    clean_item = {
        "surah": item["surah"],
        "ayah": item["ayah"],

        # Original texts
        "arabic": item["arabic"],
        "fooladvand": item["fooladvand"],
        "ansarian": item["ansarian"],

        # Clean texts for NLP
        "arabic_clean": clean_text(item["arabic"]),
        "arabic_no_diacritics": remove_diacritics(item["arabic"]),
        "fooladvand_clean": clean_text(item["fooladvand"]),
        "ansarian_clean": clean_text(item["ansarian"]),
    }

    clean_records.append(clean_item)


print("Clean records:", len(clean_records))
print(clean_records[0])

Clean records: 6236
{'surah': 1, 'ayah': 1, 'arabic': 'بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ', 'fooladvand': 'به نام خداوند رحمتگر مهربان', 'ansarian': 'به نام خدا که رحمتش بی\u200cاندازه است و مهربانی\u200cاش همیشگی.', 'arabic_clean': 'بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِیمِ', 'arabic_no_diacritics': 'بسم الله الرحمن الرحيم', 'fooladvand_clean': 'به نام خداوند رحمتگر مهربان', 'ansarian_clean': 'به نام خدا که رحمتش بی\u200cاندازه است و مهربانی\u200cاش همیشگی.'}


<div style="direction: rtl; white-space: normal; line-height: 1;">
## Save cleaned dataset

ذخیره نسخه پردازش‌شده برای مراحل بعدی RAG.
</div>

In [20]:
# Save cleaned dataset

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(
        clean_records,
        f,
        ensure_ascii=False,
        indent=2
    )


print("Saved:", output_file)
print("Records:", len(clean_records))

Saved: /Users/macbookpro/Desktop/_PROGRAMING/QuranRAG/data/processed/quran_dataset_clean.json
Records: 6236


{
  "surah": 1,
  "ayah": 1,

  "arabic": "...",
  "fooladvand": "...",
  "ansarian": "...",

  "arabic_clean": "...",
  "arabic_no_diacritics": "...",
  "fooladvand_clean": "...",
  "ansarian_clean": "..."
}

In [21]:
# Validate cleaned dataset

with open(output_file, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Records:", len(test_data))

print(test_data[0].keys())

print("\nArabic original:")
print(test_data[0]["arabic"])

print("\nArabic without diacritics:")
print(test_data[0]["arabic_no_diacritics"])

Records: 6236
dict_keys(['surah', 'ayah', 'arabic', 'fooladvand', 'ansarian', 'arabic_clean', 'arabic_no_diacritics', 'fooladvand_clean', 'ansarian_clean'])

Arabic original:
بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ

Arabic without diacritics:
بسم الله الرحمن الرحيم
